# MNIST Autoencoder Project

This notebook demonstrates a LeakyReLU Dense Autoencoder trained on the MNIST dataset. It includes data preparation, model training, evaluation under noisy conditions, and performance visualization.

## 1. Install and Import Necessary Libraries

First, we install and import the packages needed for image processing, model building, and visualization.

In [ ]:
!pip install -q torch torchvision scikit-learn matplotlib seaborn

In [ ]:
# Standard library
import os
import random

# Data & visualization
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Torchvision
import torchvision.datasets as datasets
import torchvision.transforms as transforms

## 2. Configuration & Experiment Setup

We set seeds for reproducibility and define hyperparameters such as batch size, learning rate, and dataset size.

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Configure two separate runs with a stable learning rate of 0.001
runs_config = [
    {"run_id": 1, "subset_size": 2500, "epochs": 50, "lr": 0.01},
    {"run_id": 2, "subset_size": 5000, "epochs": 100, "lr": 0.01}
]

BATCH_SIZE = 64

print("=" * 55)
print("TRAINING EXPERIMENT SETUP")
for run in runs_config:
    print(f"• Run {run['run_id']}: Dataset Size = {run['subset_size']}, Epochs = {run['epochs']}, LR = {run['lr']}")
print("=" * 55)

## 3. Load Dataset

We download the MNIST dataset via Kagglehub, falling back to torchvision if needed.

In [ ]:
def get_dataset_subset(subset_size):
    print(f"Fetching MNIST dataset directly from torchvision (subset size: {subset_size})...")
    # Download standard train split
    full_mnist = datasets.MNIST(root="./data", train=True, download=True)

    # Shuffle and select a subset deterministically using SEED
    indices = np.random.RandomState(SEED).permutation(len(full_mnist))[:subset_size]
    X = (full_mnist.data[indices].float() / 255.0).numpy()
    y = full_mnist.targets[torch.from_numpy(indices)].numpy()

    return X.reshape(-1, 28 * 28), y

## 4. Data Splits & Noise Injection

We divide the data into training (70%), validation (10%), and testing (20%) sets. Additionally, we apply a 15% noise distortion to the test set to challenge our model.

In [ ]:
def prepare_loaders(X_flat, y):
    X_temp, X_test, y_temp, y_test = train_test_split(
        X_flat, y, test_size=0.20, random_state=SEED, stratify=y
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.125, random_state=SEED, stratify=y_temp
    )

    # Inject 15% noise into the test set
    noise = np.random.normal(loc=0.0, scale=0.15, size=X_test.shape)
    X_test_noisy = np.clip(X_test + noise, 0.0, 1.0).astype(np.float32)

    train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(X_train, dtype=torch.float32))
    val_dataset   = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(X_val, dtype=torch.float32))

    train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader    = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    return train_loader, val_loader, X_train, y_train, X_test, X_test_noisy, y_test

## 5. Define LeakyReLU Dense Autoencoder

We construct a feedforward Autoencoder neural network containing Bottleneck architectures using `LeakyReLU` activations.

In [ ]:
class LeakyReLUDenseAutoencoder(nn.Module):
    def __init__(self):
        super(LeakyReLUDenseAutoencoder, self).__init__()

        self.encoder_base = nn.Sequential(
            nn.Linear(784, 128),
            nn.LeakyReLU(0.2)
        )
        self.fc_mu = nn.Linear(128, 64)
        self.fc_log_var = nn.Linear(128, 64)

        self.decoder = nn.Sequential(
            nn.Linear(64, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 784),
            nn.Sigmoid()
        )

    def forward(self, x):
        h = self.encoder_base(x)
        mu = self.fc_mu(h)
        log_var = self.fc_log_var(h)
        z = mu + torch.exp(0.5 * log_var) * torch.randn_like(mu)
        decoded = self.decoder(z)
        return decoded, mu, log_var

## 6. Model Training Loop

We initiate our training phases across the specified number of epochs, evaluating after each epoch on validation targets.

In [ ]:
run_results = {}

for run in runs_config:
    print("\n" + "="*40)
    print(f"STARTING RUN {run['run_id']}: Size={run['subset_size']}, Epochs={run['epochs']}")
    print("="*40)

    X_flat, y = get_dataset_subset(run['subset_size'])
    train_loader, val_loader, X_train, y_train, X_test, X_test_noisy, y_test = prepare_loaders(X_flat, y)

    model = LeakyReLUDenseAutoencoder().to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=run['lr'])

    train_losses = []
    val_losses = []

    for epoch in range(run['epochs']):
        model.train()
        running_train_loss = 0.0
        for data, _ in train_loader:
            data = data.to(DEVICE)
            optimizer.zero_grad()
            outputs, mu, log_var = model(data)
            recon_loss = criterion(outputs, data)
            kl_loss = -0.5 * torch.mean(1 + log_var - mu.pow(2) - log_var.exp())
            loss = recon_loss + 0.001 * kl_loss
            loss.backward()
            optimizer.step()
            running_train_loss += loss.item() * data.size(0)

        epoch_train_loss = running_train_loss / len(train_loader.dataset)
        train_losses.append(epoch_train_loss)

        model.eval()
        running_val_loss = 0.0
        with torch.no_grad():
            for data, _ in val_loader:
                data = data.to(DEVICE)
                outputs, mu, log_var = model(data)
                recon_loss = criterion(outputs, data)
                kl_loss = -0.5 * torch.mean(1 + log_var - mu.pow(2) - log_var.exp())
                loss = recon_loss + 0.001 * kl_loss
                running_val_loss += loss.item() * data.size(0)

        epoch_val_loss = running_val_loss / len(val_loader.dataset)
        val_losses.append(epoch_val_loss)

        if (epoch + 1) % (run['epochs'] // 5) == 0 or epoch == 0:
            print(f"Epoch [{epoch+1}/{run['epochs']}] - Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")

    # Evaluate on noisy test set
    model.eval()
    with torch.no_grad():
        test_noisy_tensor = torch.tensor(X_test_noisy, dtype=torch.float32).to(DEVICE)
        reconstructed, _, _ = model(test_noisy_tensor)
        reconstructed = reconstructed.cpu().numpy()

    run_results[run['run_id']] = {
        "train_losses": train_losses,
        "val_losses": val_losses,
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "X_test_noisy": X_test_noisy,
        "y_test": y_test,
        "reconstructed": reconstructed
    }

## 7. Deliverable Visualizations

We plot learning behaviors, denoising comparisons, and confusion representations of the structural output accuracy.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix

# Create a clean figure with a unified grid structure
fig = plt.figure(figsize=(24, 16))

# Helper to compute confusion matrix using normalized prototype cosine similarity
# Note: Cosine similarity normalizes vector magnitudes, preventing the L2 norm bias
# where sparse digits (like digit 1) disproportionately attract faint/noisy reconstructions.
def get_predictions(reconstructed, X_train, y_train):
    digit_prototypes = np.array([X_train[y_train == d].mean(axis=0) for d in range(10)])
    proto_norm = digit_prototypes / np.linalg.norm(digit_prototypes, axis=1, keepdims=True)
    recon_norm = reconstructed / (np.linalg.norm(reconstructed, axis=1, keepdims=True) + 1e-8)
    cosine_sim = np.dot(recon_norm, proto_norm.T)
    return np.argmax(cosine_sim, axis=1)

# Define a shared grid size of 8 rows by 16 columns
# Left 4 columns (indices 0 to 3) will be used for Metrics (Loss, CM)
# Right 12 columns (indices 4 to 15) will be used for Image Grids

# --- RUN 1 (N=2500) ---
res1 = run_results[1]

# Run 1 Loss
ax_loss1 = plt.subplot2grid((8, 16), (0, 0), rowspan=2, colspan=4)
ax_loss1.plot(res1["train_losses"], label="Train Loss", color="#1f77b4")
ax_loss1.plot(res1["val_losses"], label="Val Loss", color="#ff7f0e")
ax_loss1.set_title("Run 1: Loss Curve (N=2500)", fontweight='bold')
ax_loss1.legend()
ax_loss1.grid(True, linestyle="--", alpha=0.5)

# Run 1 Confusion Matrix
cm1 = confusion_matrix(res1["y_test"], get_predictions(res1["reconstructed"], res1["X_train"], res1["y_train"]))
ax_cm1 = plt.subplot2grid((8, 16), (2, 0), rowspan=2, colspan=4)
sns.heatmap(cm1, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax_cm1)
ax_cm1.set_title("Run 1: Confusion Matrix (Cosine Prototype)", fontweight='bold')
ax_cm1.set_xlabel("Predicted Label")
ax_cm1.set_ylabel("True Label")

# --- RUN 2 (N=5000) ---
res2 = run_results[2]

# Run 2 Loss
ax_loss2 = plt.subplot2grid((8, 16), (4, 0), rowspan=2, colspan=4)
ax_loss2.plot(res2["train_losses"], label="Train Loss", color="#2ca02c")
ax_loss2.plot(res2["val_losses"], label="Val Loss", color="#d62728")
ax_loss2.set_title("Run 2: Loss Curve (N=5000)", fontweight='bold')
ax_loss2.legend()
ax_loss2.grid(True, linestyle="--", alpha=0.5)

# Run 2 Confusion Matrix
cm2 = confusion_matrix(res2["y_test"], get_predictions(res2["reconstructed"], res2["X_train"], res2["y_train"]))
ax_cm2 = plt.subplot2grid((8, 16), (6, 0), rowspan=2, colspan=4)
sns.heatmap(cm2, annot=True, fmt="d", cmap="Greens", cbar=False, ax=ax_cm2)
ax_cm2.set_title("Run 2: Confusion Matrix (Cosine Prototype)", fontweight='bold')
ax_cm2.set_xlabel("Predicted Label")
ax_cm2.set_ylabel("True Label")

# --- VISUALIZATION GRIDS (Right Hand Side) ---
sample_indices = np.random.RandomState(SEED).choice(len(res1["X_test"]), 10, replace=False)

def plot_comparison_grid(res_dict, start_row, run_label):
    for i in range(10):
        idx = sample_indices[i]
        # Original Digit
        ax_orig = plt.subplot2grid((8, 16), (start_row, 5 + i))
        ax_orig.imshow(res_dict["X_test"][idx].reshape(28, 28), cmap='gray')
        ax_orig.axis('off')
        if i == 0:
            ax_orig.set_title(f"{run_label} Original", loc='left', fontsize=11, fontweight='bold')

        # Noisy Digit
        ax_noisy = plt.subplot2grid((8, 16), (start_row + 1, 5 + i))
        ax_noisy.imshow(res_dict["X_test_noisy"][idx].reshape(28, 28), cmap='gray')
        ax_noisy.axis('off')
        if i == 0:
            ax_noisy.set_title(f"{run_label} Noisy (15%)", loc='left', fontsize=11, fontweight='bold')

        # Reconstructed Digit
        ax_recon = plt.subplot2grid((8, 16), (start_row + 2, 5 + i))
        ax_recon.imshow(res_dict["reconstructed"][idx].reshape(28, 28), cmap='gray')
        ax_recon.axis('off')
        if i == 0:
            ax_recon.set_title(f"{run_label} Reconstructed", loc='left', fontsize=11, fontweight='bold')

# Plot both grids ensuring no index out of bounds
plot_comparison_grid(res1, 0, "Run 1 (N=2500)")
plot_comparison_grid(res2, 4, "Run 2 (N=5000)")

plt.tight_layout()
plt.show()